# 01 - Generate GeoRT Teacher Pseudo Labels

Current teacher workflow for the implemented GeoRT pipeline. It generates Metric3D, Depth Anything V2, DSINE, DMD3C, then writes separated metric and geometry teacher outputs.


In [ ]:
from pathlib import Path
import os
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = os.environ.get(
        'GEORT_PROJECT_DIR',
        '/content/drive/MyDrive/DEPTH-FUSION | Workspace/monocular_sparse_fusion/GeoRT',
    )
    PROJECT_ROOT = Path(PROJECT_DIR)
except Exception:
    PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('PROJECT_ROOT =', str(PROJECT_ROOT))


In [ ]:
%pip install -r requirements.txt


In [ ]:
import subprocess
from pathlib import Path

repos = {
    'third_party/Metric3D': ('https://github.com/YvanYin/Metric3D.git', 'hubconf.py'),
    'third_party/Depth-Anything-V2': ('https://github.com/DepthAnything/Depth-Anything-V2.git', 'depth_anything_v2/dpt.py'),
    'third_party/DSINE': ('https://github.com/baegwangbin/DSINE.git', 'projects/dsine/test_minimal.py'),
    'third_party/DMD3C': ('https://github.com/Sharpiless/DMD3C.git', 'models/BPNet.py'),
}
for dst, (url, sentinel) in repos.items():
    dst_path = Path(dst)
    if not (dst_path / sentinel).exists():
        if dst_path.exists() and sorted(p.name for p in dst_path.iterdir()) == ['.gitkeep']:
            (dst_path / '.gitkeep').unlink()
        subprocess.run(['git', 'clone', url, str(dst_path)], check=True)
    print(dst, 'OK' if (dst_path / sentinel).exists() else 'MISSING')


In [ ]:
# Required by DMD3C / BP-Net. Run on a CUDA Colab runtime.
import subprocess
import sys
from pathlib import Path

ext_dir = Path('third_party/DMD3C/exts')
if ext_dir.exists():
    subprocess.run([sys.executable, 'setup.py', 'install'], cwd=str(ext_dir), check=True)


In [ ]:
from pathlib import Path

required_dirs = [
    Path('weights/metric3d'),
    Path('weights/depth_anything_v2'),
    Path('weights/dsine'),
    Path('weights/dmd3c'),
]
for d in required_dirs:
    files = sorted([p.name for p in d.glob('*')]) if d.exists() else []
    print(d, files)
for d in required_dirs:
    assert d.exists(), f'Missing {d}' 


In [ ]:
from src.utils import load_project_config
from src.dataset import KITTIDepthCompletionDataset
from src.prepare_depth_selection import build_depth_selection_splits

CONFIG_PATH = 'configs/teacher.yaml'
REFRESH_SPLITS = True
TRAIN_COUNT = 800
SPLITS = ['train', 'val']
SAMPLE_SPLIT = 'val'
MAX_SAMPLES = None
OVERWRITE_FUSION = True

cfg, paths = load_project_config(CONFIG_PATH)
if REFRESH_SPLITS:
    counts = build_depth_selection_splits(paths['data_root'], train_count=TRAIN_COUNT)
    print('split counts:', counts)

for split in SPLITS:
    ds_tmp = KITTIDepthCompletionDataset(
        paths['data_root'], paths['split_root'], paths[f'{split}_split'], split, return_tensors=False
    )
    print('split:', split, 'samples:', len(ds_tmp))

ds = KITTIDepthCompletionDataset(
    paths['data_root'], paths['split_root'], paths[f'{SAMPLE_SPLIT}_split'], SAMPLE_SPLIT, return_tensors=False
)
sample = ds.load_sample_np(0)
print('sample:', sample['sample_id'], sample['rgb'].shape, sample['sparse'].shape, sample['K'])


In [ ]:
from pathlib import Path
from src.utils import ensure_dir, npz_has_keys

root = Path(paths['teacher_root'])
quarantine_root = root / '_quarantine' / 'invalid_depth_anything_aligned'
moved = []
for split in SPLITS:
    aligned_dir = root / 'depth_anything' / f'{split}_aligned'
    if not aligned_dir.exists():
        continue
    for p in sorted(aligned_dir.glob('*.npz')):
        if npz_has_keys(p, ['D_da_aligned', 'scale', 'shift']):
            continue
        dst_dir = ensure_dir(quarantine_root / split)
        dst = dst_dir / p.name
        if dst.exists():
            dst = dst_dir / f'{p.stem}.old{len(list(dst_dir.glob(p.stem + "*.npz")))}.npz'
        p.replace(dst)
        moved.append((str(p), str(dst)))

print('invalid aligned files quarantined:', len(moved))
for src, dst in moved[:20]:
    print(src, '->', dst)


In [ ]:
import subprocess
import sys

def run_teacher_stage(*flags, overwrite=False):
    for split in SPLITS:
        cmd = [sys.executable, '-m', 'src.teachers.generate_teachers', '--config', CONFIG_PATH, '--split', split]
        cmd.extend(flags)
        if overwrite:
            cmd.append('--overwrite')
        if MAX_SAMPLES is not None:
            cmd += ['--max_samples', str(MAX_SAMPLES)]
        print('RUN:', ' '.join(cmd))
        subprocess.run(cmd, check=True)


In [ ]:
run_teacher_stage('--run_metric3d')


In [ ]:
run_teacher_stage('--run_depth_anything')


In [ ]:
run_teacher_stage('--run_dsine')


In [ ]:
run_teacher_stage('--run_dmd3c')


In [ ]:
run_teacher_stage('--run_fusion', overwrite=OVERWRITE_FUSION)


In [ ]:
from pathlib import Path
import numpy as np

root = Path(paths['teacher_root'])
expected = {
    'metric3d': ['D_m3d'],
    'depth_anything_raw': ['D_da_raw'],
    'depth_anything_aligned': ['D_da_aligned', 'scale', 'shift'],
    'dsine': ['N_dsine'],
    'dmd3c': ['D_dmd3c'],
    'fused': ['D_teacher', 'C_teacher', 'D_full', 'C_full', 'C_dmd3c', 'w_m3d', 'w_da', 'w_dmd3c'],
    'metric_coarse': ['D_cm', 'C_cm', 'C_dmd3c', 'D_teacher', 'C_teacher'],
    'geometry_fused': ['R_G', 'C_G', 'w_da', 'w_m3d', 'w_dmd3c'],
}

def count_npz(rel):
    return len(list((root / rel).glob('*.npz')))

for split in SPLITS:
    print()
    print('SPLIT', split)
    rels = [
        f'metric3d/{split}',
        f'depth_anything/{split}_raw',
        f'depth_anything/{split}_aligned',
        f'dsine/{split}',
        f'dmd3c/{split}',
        f'fused/{split}',
        f'metric_coarse/{split}',
        f'geometry_fused/{split}',
    ]
    for rel in rels:
        print(f'{rel:34s}', count_npz(rel))

    fused_files = sorted((root / 'fused' / split).glob('*.npz'))
    if fused_files:
        with np.load(fused_files[0]) as data:
            print('first fused:', fused_files[0].name, data.files)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from src.utils import load_npz_array

split = SAMPLE_SPLIT
root = Path(paths['teacher_root'])
fused_files = sorted((root / 'fused' / split).glob('*.npz'))
assert fused_files, f'No fused outputs found for split={split}'
sid = fused_files[0].stem

ds_vis = KITTIDepthCompletionDataset(
    paths['data_root'], paths['split_root'], paths[f'{split}_split'], split, return_tensors=False
)
index_by_id = {ds_vis.samples[i].sample_id: i for i in range(len(ds_vis))}
sample = ds_vis.load_sample_np(index_by_id[sid])

D_m3d = load_npz_array(root / 'metric3d' / split / f'{sid}.npz', 'D_m3d')
D_da = load_npz_array(root / 'depth_anything' / f'{split}_aligned' / f'{sid}.npz', 'D_da_aligned')
N = load_npz_array(root / 'dsine' / split / f'{sid}.npz', 'N_dsine')
D_teacher = load_npz_array(root / 'fused' / split / f'{sid}.npz', 'D_teacher')
C_teacher = load_npz_array(root / 'fused' / split / f'{sid}.npz', 'C_teacher')
D_cm = load_npz_array(root / 'metric_coarse' / split / f'{sid}.npz', 'D_cm')
C_cm = load_npz_array(root / 'metric_coarse' / split / f'{sid}.npz', 'C_cm')
R_G = load_npz_array(root / 'geometry_fused' / split / f'{sid}.npz', 'R_G')
C_G = load_npz_array(root / 'geometry_fused' / split / f'{sid}.npz', 'C_G')

fig, ax = plt.subplots(3, 4, figsize=(20, 12))
ax[0,0].imshow(sample['rgb']); ax[0,0].set_title('RGB')
ax[0,1].imshow(sample['sparse'], cmap='magma'); ax[0,1].set_title('Sparse')
ax[0,2].imshow(D_m3d, cmap='magma'); ax[0,2].set_title('Metric3D')
ax[0,3].imshow(D_da, cmap='magma'); ax[0,3].set_title('DA aligned')
ax[1,0].imshow(((N.transpose(1,2,0) + 1) * 0.5).clip(0,1)); ax[1,0].set_title('DSINE normal')
ax[1,1].imshow(D_teacher, cmap='magma'); ax[1,1].set_title('D_teacher 1/4')
ax[1,2].imshow(C_teacher, cmap='viridis', vmin=0, vmax=1); ax[1,2].set_title('C_teacher 1/4')
ax[1,3].imshow(D_cm, cmap='magma'); ax[1,3].set_title('D_cm full')
ax[2,0].imshow(C_cm, cmap='viridis', vmin=0, vmax=1); ax[2,0].set_title('C_cm full')
ax[2,1].imshow(R_G, cmap='magma'); ax[2,1].set_title('R_G')
ax[2,2].imshow(C_G, cmap='viridis', vmin=0, vmax=1); ax[2,2].set_title('C_G')
ax[2,3].axis('off')
for a in ax.ravel():
    a.axis('off')
plt.suptitle(f'{split}/{sid}')
plt.tight_layout()
